In [1]:
import faiss
import numpy as np
import pandas as pd
import os
from collections import Counter
from sentence_transformers import SentenceTransformer

W0716 18:37:22.137000 35780 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
df = pd.read_csv("data.csv")
meta_df = df.drop('casuality', axis=1)
meta_df = meta_df.rename(columns={"img": "image_filename", "neckline": "neck"})
meta_df = meta_df.reset_index().rename(columns={"index": "faiss_id"})
meta_df.reset_index(drop=True, inplace=True)

In [3]:
model = SentenceTransformer("all-mpnet-base-v2")

In [23]:
text_fields = [
    "brand", "sleeve", "neck", "primary_color", "secondary_color",
    "fit", "pants_color", "hair_color"
]

In [38]:
def build_concat_text(df, selected_fields):
    return df[selected_fields].astype(str).apply(
        lambda row: ", ".join(f"{col} is {val}" for col, val in row.items()), axis=1
    ).tolist()

def compute_frequencies(items, fields):
    N = len(items)
    freq = {}
    for f in fields:
        cnt = Counter(item.get(f) for item in items)
        freq[f] = {v: cnt[v] / N for v in cnt}
    return freq

In [42]:
def recommend_clothes(
    meta_df,
    user_items,
    fields_to_match,
    alpha=0.5,
    beta=1.0,
    gamma=0.5,
    top_k=5,
    exclude_images=None
):
    if exclude_images is None:
        exclude_images = []

    fields_to_novel = [f for f in text_fields if f not in fields_to_match]

    # Frequency profile for novelty penalty
    freq_profiles = compute_frequencies(user_items, fields_to_novel)

    # Build FAISS index using selected similarity fields only
    data_text = build_concat_text(meta_df, fields_to_match)
    print(data_text)
    data_embeddings = model.encode(data_text, convert_to_numpy=True, show_progress_bar=True)
    faiss.normalize_L2(data_embeddings)
    index = faiss.IndexFlatIP(data_embeddings.shape[1])
    index.add(data_embeddings)

    # Create a combined query embedding from all user items
    query_texts = [" ".join([item.get(f, "") for f in fields_to_match]) for item in user_items]
    query_embs = model.encode(query_texts, convert_to_numpy=True)
    query_emb = np.mean(query_embs, axis=0, keepdims=True)
    faiss.normalize_L2(query_emb)

    # Search FAISS
    D, I = index.search(query_emb, len(meta_df))
    scores = D[0]

    results = []
    top_idx = np.argsort(scores)[::-1]

    for idx in top_idx:
        row = meta_df.iloc[idx]
        if row.image_filename in exclude_images:
            continue

        cand = {f: row.get(f) for f in text_fields}
        bonus = sum(1 for f in fields_to_match if cand[f] in {i.get(f) for i in user_items}) / len(fields_to_match) if fields_to_match else 0
        penalty = sum(freq_profiles[f].get(cand[f], 0.0) for f in fields_to_novel)
        final_score = float(scores[idx]) + alpha * bonus - beta * penalty
        results.append({
            "faiss_id": int(row.faiss_id),
            "image": row.image_filename,
            "score": final_score,
            "base_sim": float(scores[idx]),
            "bonus": bonus,
            "penalty": penalty,
            **cand
        })
        if len(results) >= top_k:
            break

    return sorted(results, key=lambda x: x["score"], reverse=True)


In [43]:
image_names = ["00000_00.jpg", "00001_00.jpg", "00860_00.jpg", "03085_00.jpg", "03567_00.jpg"]
user_items = meta_df[meta_df['image_filename'].isin(image_names)][text_fields].to_dict(orient='records')
fields_to_match = ["brand","sleeve", "neck", "hair_color"]
recs = recommend_clothes(
    meta_df=meta_df,
    user_items=user_items,
    fields_to_match=fields_to_match,
    alpha=0.8,
    beta=1.2,
    gamma=0.3,
    top_k=5
)
for r in recs:
    print(r)

["brand is levi's, sleeve is short sleeve, neck is round neck, hair_color is brown", "brand is levi's, sleeve is short sleeve, neck is round neck, hair_color is brown", 'brand is unknown, sleeve is long sleeve, neck is round neck, hair_color is blonde', 'brand is unknown, sleeve is short sleeve, neck is round neck, hair_color is black', "brand is levi's, sleeve is short sleeve, neck is round neck, hair_color is blonde", 'brand is adidas, sleeve is short sleeve, neck is round neck, hair_color is brown', 'brand is puma, sleeve is short sleeve, neck is round neck, hair_color is black', 'brand is calvin klein, sleeve is short sleeve, neck is round neck, hair_color is blonde', 'brand is adidas, sleeve is short sleeve, neck is round neck, hair_color is brown', 'brand is obey, sleeve is long sleeve, neck is round neck, hair_color is black', 'brand is unknown, sleeve is long sleeve, neck is v-neck, hair_color is brown', 'brand is adidas, sleeve is short sleeve, neck is round neck, hair_color i

Batches:   0%|          | 0/89 [00:00<?, ?it/s]

{'faiss_id': 8, 'image': '00011_00.jpg', 'score': -0.30360646724700935, 'base_sim': 0.7763935327529907, 'bonus': 0.75, 'penalty': 1.4000000000000001, 'brand': 'adidas', 'sleeve': 'short sleeve', 'neck': 'round neck', 'primary_color': 'black', 'secondary_color': 'white', 'fit': 'tight fit', 'pants_color': 'black', 'hair_color': 'brown'}
{'faiss_id': 14, 'image': '00019_00.jpg', 'score': -0.5036064672470095, 'base_sim': 0.7763935327529907, 'bonus': 0.5, 'penalty': 1.4000000000000001, 'brand': 'calvin klein', 'sleeve': 'short sleeve', 'neck': 'round neck', 'primary_color': 'black', 'secondary_color': 'white', 'fit': 'tight fit', 'pants_color': 'black', 'hair_color': 'blonde'}
{'faiss_id': 15, 'image': '00022_00.jpg', 'score': -0.6636064672470094, 'base_sim': 0.7763935327529907, 'bonus': 0.0, 'penalty': 1.2000000000000002, 'brand': 'unknown', 'sleeve': 'long sleeve', 'neck': 'v-neck', 'primary_color': 'unknown', 'secondary_color': 'unknown', 'fit': 'tight fit', 'pants_color': 'black', 'hai

In [44]:
meta_df[(meta_df['brand'] == "levi's") & (meta_df['sleeve'] == 'short sleeve') 
        & (meta_df['neck'] == 'round neck') & (meta_df['hair_color'] == 'brown')]

,faiss_id,image_filename,brand,sleeve,neck,primary_color,secondary_color,fit,pants_color,hair_color
0,0,00000_00.jpg,levi's,short sleeve,round neck,white,red,tight fit,blue,brown
1,1,00001_00.jpg,levi's,short sleeve,round neck,black,red,tight fit,black,brown
64,64,00088_00.jpg,levi's,short sleeve,round neck,white,red,tight fit,blue,brown
159,159,00214_00.jpg,levi's,short sleeve,round neck,black,pink,loose fit,black,brown
285,285,00374_00.jpg,levi's,short sleeve,round neck,white,pink,tight fit,blue,brown
660,660,00860_00.jpg,levi's,short sleeve,round neck,gray,red,tight fit,black,brown
1435,1435,01873_00.jpg,levi's,short sleeve,round neck,white,gray,tight fit,blue,brown
1873,1873,02451_00.jpg,levi's,short sleeve,round neck,white,pink,tight fit,black,brown
2348,2348,03085_00.jpg,levi's,short sleeve,round neck,white,red,loose fit,blue,brown
2732,2732,03567_00.jpg,levi's,short sleeve,round neck,white,yellow,tight fit,yellow,brown
